In [0]:
# The project has no requirements.lock yet; this one-off module only needs openpyxl for the report workbook
%pip install openpyxl --quiet
dbutils.library.restartPython()

In [0]:
# s00_build_a_score_snapshot: ONE-OFF. Freeze the current production A-Score as a benchmark for the cut-off analyses.
# STEP 0 = read-only report (no writes). STEP 1 runs only after the owner confirms Step 0 (STEP1_CONFIRMED below).
# Writes (Step 1 only): one table <catalog>.<schema>.acard_a_score_snapshot_<yyyymmdd>,
#   <artifacts_root>/reports/a_score_snapshot.xlsx and reports/run_summaries/s00_build_a_score_snapshot.json.
# Never writes to the source table or to acard_master.
import sys, os, json, hashlib, importlib, shutil, tempfile, re
from datetime import datetime, timezone
sys.path.insert(0, os.getcwd())          # notebook lives in src/
import common
importlib.reload(common)
PROJECT_ROOT = common.find_project_root(os.getcwd())

import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score

CFG_PATH = os.path.join(PROJECT_ROOT, "config", "config.yaml")
cfg = common.load_config(CFG_PATH)
SEED = cfg["random_seed"]
np.random.seed(SEED)
CONFIG_SHA = hashlib.sha256(open(CFG_PATH, "rb").read()).hexdigest()
GIT_COMMIT = common.git_commit(PROJECT_ROOT)

CAT, SCH = cfg["databricks"]["catalog"], cfg["databricks"]["schema"]
MASTER_TBL = f"{CAT}.{SCH}.{cfg['tables']['master']}"
SOURCE_TBL = "mis_automation.silver_credit_card.cc_a_score_adhoc_data_file"   # owner request, READ-ONLY
JOIN_KEY = "CRNNO"                                   # owner request: CRNNO in both tables
ID_COL = cfg["columns"]["id"]                        # project app_id (folds / OOF use app_id = this column)
DATE_COL = cfg["columns"]["date"]                    # application date (APPRECDDATE, drives the windows)
DECISION_COL = cfg["columns"]["decision_time"]       # APPROVEDATE
STATUS_COL = cfg["columns"]["status"]
LABELLED = list(cfg["labelled_statuses"])
WINDOWS = list(cfg["windows"])                       # DEV, OOT, FWD, STRESS
ARTIFACTS = common.artifacts_path(cfg)
REG_PATH = os.path.join(ARTIFACTS, "model_registry.json")
REPORTS = os.path.join(ARTIFACTS, "reports")
RUNSUM = os.path.join(REPORTS, "run_summaries")
REPORT_PATH = os.path.join(REPORTS, "a_score_snapshot.xlsx")
RUN_TS = datetime.now(timezone.utc)
SNAP_TBL = f"{CAT}.{SCH}.acard_a_score_snapshot_{RUN_TS:%Y%m%d}"
assert SNAP_TBL not in (SOURCE_TBL, MASTER_TBL)

def norm_key(s: pd.Series) -> pd.Series:
    """Owner rule: cast to string, trim, strip a trailing '.0' (float conversion), strip leading zeros
    (owner 2026-11-18: master CRNNO is zero-padded to 9 digits, source is bigint), '' -> null."""
    out = s.astype("string").str.strip().str.replace(r"\.0$", "", regex=True).str.lstrip("0")
    return out.mask(out == "", pd.NA)

assert norm_key(pd.Series([" 123.0 ", "123", "", None, 1230.0, "10.05", "077553000", 77553000])).tolist() == \
    ["123", "123", pd.NA, pd.NA, "1230", "10.05", "77553000", "77553000"], "norm_key self-test failed"

def delta_version(tbl):
    return int(spark.sql(f"DESCRIBE HISTORY {tbl} LIMIT 1").collect()[0]["version"])

VERSIONS_BEFORE = {t: delta_version(t) for t in (SOURCE_TBL, MASTER_TBL)}
print(f"git commit {GIT_COMMIT} | config sha256 {CONFIG_SHA[:12]} | id column {ID_COL}")
print(f"source {SOURCE_TBL} v{VERSIONS_BEFORE[SOURCE_TBL]} | master {MASTER_TBL} v{VERSIONS_BEFORE[MASTER_TBL]}")
print(f"snapshot target (Step 1 only): {SNAP_TBL}")

In [0]:
src_sp = spark.table(SOURCE_TBL)
SRC_TYPES = dict(src_sp.dtypes)
src = src_sp.toPandas()
N_SRC = len(src)
print(f"{SOURCE_TBL}: {N_SRC:,} rows, {len(src.columns)} columns")

prof = []
for c in src.columns:
    s = src[c]
    prof.append({"column": c, "type": SRC_TYPES[c], "non_null": int(s.notna().sum()), "null": int(s.isna().sum()),
                 "distinct": int(s.nunique()), "min": str(s.min()), "max": str(s.max())})   # str: mixed types per column
SRC_PROFILE = pd.DataFrame(prof)
display(SRC_PROFILE)

SRC_SAMPLE = src.sort_values(JOIN_KEY).head(5)
display(SRC_SAMPLE)

# Date column check: BASE granularity (day of month, time part)
if "BASE" in src.columns:
    b = pd.to_datetime(src["BASE"])
    print("BASE day-of-month values:", sorted(b.dt.day.unique().tolist()),
          "| has time part:", bool((b.dt.normalize() != b).any()),
          "| months:", b.dt.to_period("M").nunique(), f"({b.min():%Y-%m} .. {b.max():%Y-%m})")

# Proposed roles (for owner confirmation)
score_like = [c for c in src.columns if re.search("SCORE", c, re.I)]
date_like = [c for c, t in SRC_TYPES.items() if t in ("timestamp", "date")]
appid_like = [c for c in src.columns if re.search("APP|REF", c, re.I)]
version_like = [c for c in src.columns if re.search("VER|MODEL", c, re.I)]
PROPOSAL = {"a_score": score_like, "score_date": date_like, "application_id": appid_like, "score_version": version_like}
print("Proposed roles:", PROPOSAL)

In [0]:
src["crn_norm"] = norm_key(src[JOIN_KEY])

m_cols = spark.table(MASTER_TBL).columns
keep = [c for c in dict.fromkeys([ID_COL, JOIN_KEY, "CREDITCARDNO", "Reference_Number", "CIF", DATE_COL, DECISION_COL, "window", STATUS_COL]) if c in m_cols]
mp = spark.table(MASTER_TBL).select(*keep).toPandas()
# window access through common.load_window (OOT read lock: raises unless the registry is locked)
parts = [common.load_window(mp, w, cfg, registry_path=REG_PATH) for w in WINDOWS]
parts.append(mp[~mp["window"].isin(WINDOWS)].copy())          # OUT_OF_SCOPE / null window
apps = pd.concat(parts, ignore_index=True)
assert len(apps) == len(mp)
apps["window"] = apps["window"].fillna("NULL").astype(str)
apps["crn_norm"] = norm_key(apps[JOIN_KEY])
apps["app_date"] = pd.to_datetime(apps[DATE_COL])
print(f"{MASTER_TBL}: {len(apps):,} rows")

def dup_stats(df, key, label):
    vc = df[key].dropna().value_counts()
    return {"table": label, "rows": len(df), "null_key": int(df[key].isna().sum()), "distinct_key": int(len(vc)),
            "keys_with_gt1_rows": int((vc > 1).sum()), "rows_in_dup_keys": int(vc[vc > 1].sum()), "max_rows_per_key": int(vc.max()) if len(vc) else 0}

rows = [dup_stats(src, "crn_norm", "source")]
rows.append(dup_stats(apps, "crn_norm", "acard_master ALL windows"))
for w in WINDOWS + sorted(set(apps["window"]) - set(WINDOWS)):
    rows.append(dup_stats(apps[apps["window"] == w], "crn_norm", f"acard_master {w}"))
oot_gb = apps[(apps["window"] == "OOT") & apps[STATUS_COL].isin(LABELLED)]
rows.append(dup_stats(oot_gb, "crn_norm", "acard_master OOT GOOD+BAD"))
UNIQ = pd.DataFrame(rows)
display(UNIQ)

# Is any other master column a unique application key? (one row per application needs one)
cand = []
for c in [c for c in ["CREDITCARDNO", "Reference_Number"] if c in apps.columns]:
    k = norm_key(apps[c])
    cand.append({"column": c, "null": int(k.isna().sum()), "distinct": int(k.nunique()), "rows": len(k),
                 "unique_non_null": bool(k.dropna().is_unique)})
KEY_CANDIDATES = pd.DataFrame(cand)
display(KEY_CANDIDATES)

In [0]:
def differing_columns(df, key, cols, label):
    d = df[df[key].notna() & df.duplicated(key, keep=False)]
    n_keys = d[key].nunique()
    out = []
    for c in cols:
        n_diff = int((d.groupby(key)[c].nunique(dropna=False) > 1).sum()) if n_keys else 0
        out.append({"table": label, "column": c, "dup_keys": int(n_keys), "dup_keys_where_column_differs": n_diff})
    return out

src_cols = [c for c in src.columns if c not in (JOIN_KEY, "crn_norm")]
m_cmp = [c for c in ["CREDITCARDNO", "Reference_Number", "CIF", DATE_COL, DECISION_COL, "window", STATUS_COL] if c in apps.columns]
DIFF = pd.DataFrame(differing_columns(src, "crn_norm", src_cols, "source")
                    + differing_columns(apps, "crn_norm", m_cmp, "acard_master ALL windows"))
display(DIFF)

# master: day gap between the first and the last application of the same CRNNO
d = apps[apps["crn_norm"].notna() & apps.duplicated("crn_norm", keep=False)]
if len(d):
    gap = d.groupby("crn_norm")["app_date"].agg(lambda s: (s.max() - s.min()).days)
    print("master dup CRNNO: days between first and last application:", gap.describe().round(1).to_dict())
    display(d.sort_values(["crn_norm", "app_date"]).head(10))

In [0]:
src_one = src.dropna(subset=["crn_norm"])
SRC_KEYS = set(src_one["crn_norm"])
apps["matched"] = apps["crn_norm"].isin(SRC_KEYS)

# timing of the matched score vs the application (shown for the Step 1 rule, not applied here)
m1 = apps[apps["matched"]].merge(src_one[["crn_norm", "BASE"]], on="crn_norm", how="left")
m1["base_on_or_before_app"] = pd.to_datetime(m1["BASE"]) <= m1["app_date"]
m1["month_offset"] = (pd.to_datetime(m1["BASE"]).dt.to_period("M") - m1["app_date"].dt.to_period("M")).apply(lambda x: x.n)

def cov(df, by):
    g = df.groupby(by, dropna=False)
    out = g.agg(applications=("matched", "size"), matched=("matched", "sum")).reset_index()
    out["coverage"] = (out["matched"] / out["applications"]).round(4)
    return out

COV_WINDOW = cov(apps, "window")
COV_WINDOW = pd.concat([COV_WINDOW, pd.DataFrame([{"window": "ALL", "applications": len(apps), "matched": int(apps["matched"].sum()),
                                                   "coverage": round(apps["matched"].mean(), 4)}])], ignore_index=True)
tmp = m1.groupby("window").agg(matched=("base_on_or_before_app", "size"), base_on_or_before_app=("base_on_or_before_app", "sum")).reset_index()
COV_WINDOW = COV_WINDOW.merge(tmp[["window", "base_on_or_before_app"]], on="window", how="left")
display(COV_WINDOW)

oot = apps[apps["window"] == "OOT"]
COV_OOT_STATUS = cov(oot, STATUS_COL)
gbm = oot[oot[STATUS_COL].isin(LABELLED)]
COV_OOT_STATUS = pd.concat([COV_OOT_STATUS, pd.DataFrame([{STATUS_COL: "GOOD+BAD", "applications": len(gbm), "matched": int(gbm["matched"].sum()),
                                                           "coverage": round(gbm["matched"].mean(), 4)}])], ignore_index=True)
display(COV_OOT_STATUS)

# coverage by application month (where do the unmatched sit?)
apps["app_month_s"] = apps["app_date"].dt.to_period("M").astype(str)
COV_MONTH = cov(apps, ["window", "app_month_s"])
display(COV_MONTH)

# BASE month minus application month, matched applications
OFFSET = m1.groupby(["window", "month_offset"]).size().unstack(fill_value=0)
display(OFFSET.reset_index())

In [0]:
SCORE_COL = "FINAL_SCORE"           # proposed in (a); confirm
d_oot = oot[oot[STATUS_COL].isin(LABELLED) & oot["matched"]].merge(src_one[["crn_norm", SCORE_COL, "BASE"]], on="crn_norm", how="left")
d_oot = d_oot[d_oot[SCORE_COL].notna()].copy()
d_oot["y"] = (d_oot[STATUS_COL] == "BAD").astype(int)

def deciles(df, label):
    df = df.copy()
    df["decile"] = pd.qcut(df[SCORE_COL].rank(method="first"), 10, labels=False) + 1      # 1 = lowest score
    t = df.groupby("decile").agg(n=("y", "size"), bads=("y", "sum"), score_min=(SCORE_COL, "min"), score_max=(SCORE_COL, "max")).reset_index()
    t["bad_rate"] = (t["bads"] / t["n"]).round(4)
    t.insert(0, "subset", label)
    rho = spearmanr(t["decile"], t["bad_rate"]).correlation
    auc = roc_auc_score(df["y"], -df[SCORE_COL])        # > 0.5 if higher score = lower risk
    return t, {"subset": label, "n": len(df), "bads": int(df["y"].sum()), "spearman_decile_vs_bad_rate": round(rho, 4),
               "auc_higher_is_lower_risk": round(auc, 4), "gini": round(2 * auc - 1, 4)}

t_all, s_all = deciles(d_oot, "OOT GOOD+BAD, all matched")
sub = d_oot[pd.to_datetime(d_oot["BASE"]) <= d_oot["app_date"]]
t_pre, s_pre = deciles(sub, "OOT GOOD+BAD, BASE on or before application date") if len(sub) >= 100 and sub["y"].sum() >= 10 else (pd.DataFrame(), {"subset": "BASE on or before application date", "n": len(sub), "bads": int(sub["y"].sum())})
DIRECTION = pd.concat([t_all, t_pre], ignore_index=True)
DIRECTION_SUMMARY = pd.DataFrame([s_all, s_pre])
display(DIRECTION)
display(DIRECTION_SUMMARY)
HIGHER_IS_LOWER_RISK = bool(s_all["auc_higher_is_lower_risk"] > 0.5 and s_all["spearman_decile_vs_bad_rate"] < 0)
print("Direction: higher A-Score = lower risk ->", HIGHER_IS_LOWER_RISK)

In [0]:
# Step 0 ends here: nothing has been written. Delta versions must be unchanged.
assert {t: delta_version(t) for t in VERSIONS_BEFORE} == VERSIONS_BEFORE, "source or master changed during Step 0"
print("Step 0 complete, no writes. Source and master Delta versions unchanged:", VERSIONS_BEFORE)

# ---- Owner confirmation (set after reviewing Step 0) ----
STEP1_CONFIRMED = True                   # owner confirmed 2026-11-18
# owner 2026-11-18: app_id = CRNNO; where CRNNO is duplicated in acard_master use CREDITCARDNO if available, otherwise dedup
APP_ID_COL = "CRNNO (CREDITCARDNO where CRNNO is duplicated, else dedup)"
CARD_COL = "CREDITCARDNO"
SCORE_DATE_COL = "BASE"                  # proposed in (a)
SCORE_VERSION_COL = None                 # none found in the source
MULTI_MATCH_RULE = "if several source rows match an application, keep the one whose score date is on or before the application date and closest to it"

In [0]:
assert STEP1_CONFIRMED, "STOP: Step 1 runs only after the owner confirms Step 0 (set STEP1_CONFIRMED = True in the gate cell)"
assert APP_ID_COL, "STOP: set APP_ID_COL to the confirmed application key"
assert SCORE_DATE_COL in src.columns, "STOP: the source has no score date column -> ask the owner (no rule for multiple matches)"

# app_id rule (owner): CRNNO; for duplicated CRNNO use CREDITCARDNO if available; otherwise dedup (keep earliest application)
a = apps[[JOIN_KEY, CARD_COL, "crn_norm", "app_date", "window", STATUS_COL]].copy()
a["crn_raw"] = a[JOIN_KEY].astype("string").str.strip()          # master CRNNO as stored (zero-padded), for joining back
a["card_norm"] = a[CARD_COL].astype("string").str.strip().str.replace(r"\.0$", "", regex=True)
a["card_norm"] = a["card_norm"].mask(a["card_norm"] == "", pd.NA)
dup_crn = a["crn_norm"].notna() & a.duplicated("crn_norm", keep=False)
use_card = (dup_crn | a["crn_norm"].isna()) & a["card_norm"].notna()
a["app_id"] = a["crn_raw"].where(~use_card, a["card_norm"])
a["app_id_source"] = np.where(use_card, "CREDITCARDNO (CRNNO duplicated)", "CRNNO")
assert a["app_id"].notna().all(), "STOP: applications with neither CRNNO nor CREDITCARDNO"
assert not (set(a.loc[use_card, "app_id"]) & set(a.loc[~use_card, "app_id"])), "STOP: a CREDITCARDNO equals a CRNNO app_id"
n_before = len(a)
a = a.sort_values(["app_id", "app_date"]).drop_duplicates("app_id", keep="first").reset_index(drop=True)
N_DEDUP_DROPPED = n_before - len(a)
APP_ID_RULE = (a["app_id_source"].value_counts().rename_axis("app_id_source").reset_index(name="applications"))
APP_ID_RULE = pd.concat([APP_ID_RULE, pd.DataFrame([{"app_id_source": "rows removed by dedup (duplicated CRNNO, no CREDITCARDNO)", "applications": N_DEDUP_DROPPED}])], ignore_index=True)
display(APP_ID_RULE)
assert a["app_id"].is_unique

cand = a.merge(src_one[["crn_norm", SCORE_COL, SCORE_DATE_COL] + ([SCORE_VERSION_COL] if SCORE_VERSION_COL else [])],
               on="crn_norm", how="inner")
cand["score_date"] = pd.to_datetime(cand[SCORE_DATE_COL])
n_cand = cand.groupby("app_id")["crn_norm"].transform("size")
single = cand[n_cand == 1].copy()
# one source row: kept as specified; timing recorded in match_rule (score month can be after the application)
single["match_rule"] = np.where(single["score_date"] <= single["app_date"], "SINGLE_SOURCE_ROW_SCORE_ON_OR_BEFORE_APP_DATE",
                                "SINGLE_SOURCE_ROW_SCORE_AFTER_APP_DATE")
multi = cand[n_cand > 1]
multi_ok = multi[multi["score_date"] <= multi["app_date"]]
multi_ok = (multi_ok.assign(gap=(multi_ok["app_date"] - multi_ok["score_date"]).dt.days)
            .sort_values(["app_id", "gap", "score_date"], ascending=[True, True, False])
            .drop_duplicates("app_id").drop(columns="gap").assign(match_rule="MULTI_CLOSEST_ON_OR_BEFORE_APP_DATE"))
MULTI_NONE_BEFORE = sorted(set(multi["app_id"]) - set(multi_ok["app_id"]))
snap = pd.concat([single, multi_ok], ignore_index=True)

snap_out = pd.DataFrame({
    "app_id": snap["app_id"].astype(str),
    "CRNNO": snap["crn_raw"].astype(str),
    "a_score": pd.to_numeric(snap[SCORE_COL]).astype("int64"),
    "score_date": snap["score_date"].dt.date,
    "score_version": snap[SCORE_VERSION_COL].astype(str) if SCORE_VERSION_COL else None,
    "match_rule": snap["match_rule"],
    "source_table": SOURCE_TBL,
    "built_at": pd.Timestamp(RUN_TS).tz_convert(None),
})
assert snap_out["app_id"].is_unique

a["in_snapshot"] = a["app_id"].isin(set(snap_out["app_id"]))
a["unmatched_reason"] = np.where(a["in_snapshot"], None,
                         np.where(a["crn_norm"].isna(), "NULL_CRNNO",
                         np.where(a["app_id"].isin(MULTI_NONE_BEFORE), "MULTI_NO_SCORE_ON_OR_BEFORE_APP_DATE", "CRNNO_NOT_IN_SOURCE")))
N_MATCHED = int(a["in_snapshot"].sum())
RULE_COUNTS = snap_out["match_rule"].value_counts().rename_axis("match_rule").reset_index(name="applications")
print(f"applications {len(a):,} | matched {N_MATCHED:,} ({N_MATCHED / len(a):.2%}) | multi with no score on/before: {len(MULTI_NONE_BEFORE)}")
display(RULE_COUNTS)

In [0]:
assert STEP1_CONFIRMED
assert SNAP_TBL not in (SOURCE_TBL, MASTER_TBL) and SNAP_TBL.split(".")[-1].startswith("acard_a_score_snapshot_")
from pyspark.sql.types import StructType, StructField, StringType, LongType, DateType, TimestampType
SNAP_SCHEMA = StructType([StructField("app_id", StringType(), False), StructField("CRNNO", StringType(), True),
                          StructField("a_score", LongType(), False), StructField("score_date", DateType(), True),
                          StructField("score_version", StringType(), True), StructField("match_rule", StringType(), False),
                          StructField("source_table", StringType(), False), StructField("built_at", TimestampType(), False)])
def _py(v):
    """pandas/numpy value -> plain Python value for Spark (tz-naive Timestamps are UTC)."""
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return None
    if isinstance(v, pd.Timestamp):
        return (v if v.tzinfo else v.tz_localize("UTC")).to_pydatetime()
    if isinstance(v, np.integer):
        return int(v)
    return v
_rows = [tuple(_py(v) for v in r) for r in snap_out.astype(object).where(snap_out.notna(), None).itertuples(index=False, name=None)]
(spark.createDataFrame(_rows, schema=SNAP_SCHEMA)
      .write.mode("overwrite").option("overwriteSchema", "true")
      .saveAsTable(SNAP_TBL))
spark.sql(f"COMMENT ON TABLE {SNAP_TBL} IS 'Frozen A-Score benchmark ({SOURCE_TBL}, {SCORE_COL}) for acard_master applications, built by src/s00_build_a_score_snapshot at {RUN_TS:%Y-%m-%dT%H:%M:%SZ}, git {GIT_COMMIT}. Rule: {MULTI_MATCH_RULE}.'")
print(f"written {SNAP_TBL}: {spark.table(SNAP_TBL).count():,} rows")

In [0]:
assert STEP1_CONFIRMED
os.makedirs(REPORTS, exist_ok=True)
unm = a[~a["in_snapshot"]]
UNMATCHED_BY_WINDOW = (a.groupby("window").agg(applications=("app_id", "size"), matched=("in_snapshot", "sum")).reset_index()
                       .merge(unm.groupby(["window", "unmatched_reason"]).size().unstack(fill_value=0).reset_index(), on="window", how="left").fillna(0))
UNMATCHED_BY_WINDOW["unmatched"] = UNMATCHED_BY_WINDOW["applications"] - UNMATCHED_BY_WINDOW["matched"]
UNMATCHED_BY_WINDOW["coverage"] = (UNMATCHED_BY_WINDOW["matched"] / UNMATCHED_BY_WINDOW["applications"]).round(4)
UNMATCHED_LIST = unm[["window", "app_id", "app_id_source", JOIN_KEY, CARD_COL, "app_date", STATUS_COL, "unmatched_reason"]].sort_values(["window", "app_date"])

meta = pd.DataFrame({"key": ["module", "run_time_utc", "git_commit", "config_sha256", "source_table", "source_version", "master_table",
                             "master_version", "snapshot_table", "join_key", "key_normalisation", "app_id_column", "score_column",
                             "score_date_column", "score_version_column", "multi_match_rule"],
                     "value": ["s00_build_a_score_snapshot (one-off)", f"{RUN_TS:%Y-%m-%dT%H:%M:%SZ}", GIT_COMMIT, CONFIG_SHA, SOURCE_TBL,
                               VERSIONS_BEFORE[SOURCE_TBL], MASTER_TBL, VERSIONS_BEFORE[MASTER_TBL], SNAP_TBL, JOIN_KEY,
                               "cast to string, trim, strip trailing '.0', strip leading zeros, '' -> null (both sides)", APP_ID_COL, SCORE_COL,
                               SCORE_DATE_COL, SCORE_VERSION_COL or "none", MULTI_MATCH_RULE]})
MATCH_RULE_SHEET = pd.concat([pd.DataFrame([{"match_rule": "RULE", "applications": MULTI_MATCH_RULE}]), RULE_COUNTS,
                              pd.DataFrame([{"match_rule": "MULTI_NO_SCORE_ON_OR_BEFORE_APP_DATE (unmatched)", "applications": len(MULTI_NONE_BEFORE)}])], ignore_index=True)
sheets = {"meta": meta, "source_columns": SRC_PROFILE, "source_sample": SRC_SAMPLE.drop(columns="crn_norm", errors="ignore"),
          "uniqueness": UNIQ, "key_candidates": KEY_CANDIDATES, "dup_columns_differ": DIFF, "coverage_by_window": COV_WINDOW,
          "coverage_oot_status": COV_OOT_STATUS, "coverage_by_month": COV_MONTH, "score_month_offset": OFFSET.reset_index(),
          "direction_deciles": DIRECTION, "direction_summary": DIRECTION_SUMMARY, "app_id_rule": APP_ID_RULE, "match_rule": MATCH_RULE_SHEET,
          "unmatched_by_window": UNMATCHED_BY_WINDOW, "unmatched_applications": UNMATCHED_LIST}
with tempfile.NamedTemporaryFile(suffix=".xlsx", delete=False) as tf:
    with pd.ExcelWriter(tf.name, engine="openpyxl") as xw:
        for nm, df in sheets.items():
            df.to_excel(xw, sheet_name=nm[:31], index=False)
    shutil.copyfile(tf.name, REPORT_PATH)          # copyfile, not copy2 (UC Volume)
print("report:", REPORT_PATH, "| sheets:", list(sheets))
display(UNMATCHED_BY_WINDOW)

In [0]:
assert STEP1_CONFIRMED
st = spark.table(SNAP_TBL)
n_rows = st.count()
checks = {
    "snapshot columns as specified": st.columns == ["app_id", "CRNNO", "a_score", "score_date", "score_version", "match_rule", "source_table", "built_at"],
    "app_id unique in the snapshot": st.select("app_id").distinct().count() == n_rows,
    "no null app_id / a_score": st.filter("app_id IS NULL OR a_score IS NULL").count() == 0,
    "row count = matched applications": n_rows == N_MATCHED,
    "every snapshot app_id is an acard_master application": set(r[0] for r in st.select("app_id").collect()) <= set(a["app_id"]),
    "coverage reported (by window, OOT GOOD/BAD)": len(COV_WINDOW) > 0 and len(COV_OOT_STATUS) > 0,
    "report exists with unmatched-by-window and match-rule sheets": os.path.exists(REPORT_PATH)
        and {"unmatched_by_window", "unmatched_applications", "match_rule"} <= set(pd.ExcelFile(REPORT_PATH).sheet_names),
    "unmatched list = applications - matched": len(UNMATCHED_LIST) == len(a) - N_MATCHED,
    "applications = master rows - dedup drops": len(a) == len(apps) - N_DEDUP_DROPPED,
    "one snapshot row per matched master row (dedup aside)": n_rows == int(a["crn_norm"].isin(SRC_KEYS).sum()),
    "source and acard_master not written": {t: delta_version(t) for t in VERSIONS_BEFORE} == VERSIONS_BEFORE,
}
for k, v in checks.items():
    print("PASS" if v else "FAIL", "-", k)

os.makedirs(RUNSUM, exist_ok=True)
summary = {"module": "s00_build_a_score_snapshot", "run_time_utc": f"{RUN_TS:%Y-%m-%dT%H:%M:%SZ}", "git_commit": GIT_COMMIT,
           "config_sha256": CONFIG_SHA, "snapshot_table": SNAP_TBL, "source_table": SOURCE_TBL, "applications": len(a),
           "matched": N_MATCHED, "dedup_dropped": N_DEDUP_DROPPED,
           "app_id_rule": APP_ID_RULE.set_index("app_id_source")["applications"].to_dict(), "coverage": round(N_MATCHED / len(a), 4), "match_rule_counts": RULE_COUNTS.set_index("match_rule")["applications"].to_dict(),
           "direction": DIRECTION_SUMMARY.to_dict("records"), "checks": checks, "all_pass": all(checks.values())}
with open(os.path.join(RUNSUM, "s00_build_a_score_snapshot.json"), "w") as f:
    json.dump(summary, f, indent=2, default=str)
failed = [k for k, v in checks.items() if not v]
assert not failed, f"acceptance failed: {failed}"
print("All acceptance checks PASS")